In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import ParameterGrid
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.base import BaseEstimator, TransformerMixin
from tqdm import tqdm
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor

In [2]:
df_final = pd.read_excel("data/processed/df_final.xlsx")
df_final = df_final.sort_values("Date").reset_index(drop=True) 

In [3]:
covariates = ["dolvol_lag2", "maxret", "retvol", "mom36m", "mom12m", "mom6m", "mom1m", "chmom", "turn", "indmom", "baspread", "illiq", "stdturn", "beta", "beta_squared",
"idiovol", "mvel1", "agr", "cashpr", "chinv", "chsh", "depr", "dy", "ep", "invest", "rd_mve", "sp", "nincr"]

In [4]:
def generate_time_splits(df, date_col='Date',
                                   val_months=12,
                                   test_months=12,
                                   step_months=12,
                                   min_train_months=192):
    """
    Génère des splits temporels:
    - Train cumulatif (augmente d'un an à chaque refit)
    - Train 306 mois (= 85% de la data)
    - Validation = fenêtre fixe glissante de 1 an
    - Test = 1 an 
    - Avance de step_months à chaque itération : 1 an

    Paramètres :
    - df : DataFrame trié par date
    - date_col : nom de la colonne des dates
    - val_months : taille de la validation
    - test_months : taille du test 
    - step_months : pas de glissement
    - min_train_months : nombre minimum de mois de train initial

    Retour :
    - splits : liste de tuples (train_idx, val_idx, test_idx)
    """

    #On coupe chronologiquement donc on trie par date 
    df = df.sort_values(date_col).reset_index(drop=True)
    dates = sorted(df[date_col].unique())
    total_months = len(dates)

    splits = []

    #On démarre après avoir au moins min_train_months pour le train
    start = min_train_months
    while True:
        train_end = start  # train va de 0 jusqu'à train_end
        val_start = train_end
        val_end = val_start + val_months
        test_start = val_end
        test_end = test_start + test_months

        #Stop quand on a plus assez pour test
        if test_end > total_months:
            break

        train_dates = dates[:train_end]  
        val_dates = dates[val_start:val_end]
        test_dates = dates[test_start:test_end]

        train_idx = df[df[date_col].isin(train_dates)].index.tolist()
        val_idx = df[df[date_col].isin(val_dates)].index.tolist()
        test_idx = df[df[date_col].isin(test_dates)].index.tolist()

        splits.append((train_idx, val_idx, test_idx))

        #Décale fenêtre de un → on réactualise tous les 1 ans
        start += step_months

    return splits

In [5]:
def preprocess_split(X_train, X_val, X_test, covariates):
    """
    Impute les NaN par moyenne par Ticker (fit sur train),
    puis normalise chaque covariable entre -1 et 1 par date (rang cross-sectionnel).

    Paramètres
    ----------
    X_train, X_val, X_test : DataFrames bruts (avec 'Ticker' et 'Date')
    covariates : liste des colonnes numériques à traiter

    Retour
    ------
    X_train_scaled, X_val_scaled, X_test_scaled : DataFrames transformés (covariates seulement)
    """
    #Gestion des NaN : moyenne par Ticker calculée sur le train
    means_by_ticker = x_train.groupby("Ticker")[covariates].mean(numeric_only=True)

    def fill_na_with_means(df):
        df = df.copy()
        for col in covariates:
            #On remplace NaN par la moyenne du ticker
            df[col] = df.apply(
                lambda row: means_by_ticker[col][row["Ticker"]] 
                            if pd.isna(row[col]) and row["Ticker"] in means_by_ticker.index 
                            else row[col],
                axis=1
            )
        return df

    x_train_imp = fill_na_with_means(x_train)
    x_val_imp   = fill_na_with_means(x_val)
    x_test_imp  = fill_na_with_means(x_test)

    #Normalisation cross-sectionnelle : par date
    def normalize_by_date(df):
        df = df.copy()
        out = pd.DataFrame(index=df.index, columns=covariates)
        for date_key, group_idx in df.groupby("Date").groups.items():
            sub = df.loc[group_idx, covariates]
            for cov in covariates:
                temp = sub[cov].dropna().sort_values()
                n = len(temp)
                if n == 1:
                    scores = pd.Series([0.0], index=temp.index)
                else:
                    scores = pd.Series(
                        2 * np.arange(n) / (n - 1) - 1, index=temp.index
                    )
                out.loc[temp.index, cov] = scores
        return out.astype(float)

    x_train_scaled = normalize_by_date(x_train_imp)
    x_val_scaled   = normalize_by_date(x_val_imp)
    x_test_scaled  = normalize_by_date(x_test_imp)

    #garde l'information de la date et du ticker
    x_train_scaled = pd.concat([x_train_imp[['Ticker','Date']].reset_index(drop=True), x_train_scaled.reset_index(drop=True)], axis=1)
    x_val_scaled   = pd.concat([x_val_imp[['Ticker','Date']].reset_index(drop=True), x_val_scaled.reset_index(drop=True)], axis=1)
    x_test_scaled  = pd.concat([x_test_imp[['Ticker','Date']].reset_index(drop=True), x_test_scaled.reset_index(drop=True)], axis=1)

    return x_train_scaled, x_val_scaled, x_test_scaled

In [6]:
#R²
#Mesures : → peut être à tej 
def r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum(y_true**2) 
    return 1 - ss_res/ss_tot if ss_tot != 0 else np.nan

#% ratio:
def success_ratio(y_true, y_pred, ignore_zero=True):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    sign_true = np.sign(y_true)
    sign_pred = np.sign(y_pred)

    if ignore_zero:
        mask = sign_true != 0
        sign_true = sign_true[mask]
        sign_pred = sign_pred[mask]

    if len(sign_true) == 0:
        return np.nan  
    return (sign_true == sign_pred).mean()

#R2 benchmark 
def r2_vs_benchmark(y_true, y_pred_model, y_pred_bench):
    T = len(y_true)
    mspe_model = (1/T) * np.sum((y_true - y_pred_model)**2)
    mspe_bench = (1/T) * np.sum((y_true - y_pred_bench)**2)
    return 1 - (mspe_model / mspe_bench)

In [7]:
#Permet de récupérer x et y 
def get_x_y(df, idx, target="excess_return"):
    subset = df.loc[idx].copy()
    x = subset.drop(columns=[target]) #garde toutes les colonnes mais enlève excess return
    y = subset[target]
    return x, y

In [8]:
#On découpe les splits puis on applique la gestion des NaN et la normalisation définie plus haut
splits = generate_time_splits(df_final)

preprocessed_splits = []

for train_idx, val_idx, test_idx in tqdm(splits):
    x_train, y_train = get_x_y(df_final, train_idx)
    x_val, y_val = get_x_y(df_final, val_idx)
    x_test, y_test = get_x_y(df_final, test_idx)

    #Imputation + Normalisation
    x_train, x_val, x_test = preprocess_split(x_train, x_val, x_test, covariates) #on enlève ticker et date

    preprocessed_splits.append((x_train, y_train, x_val, y_val, x_test, y_test))

100%|██████████| 13/13 [02:00<00:00,  9.28s/it]


In [9]:
"""HISTORICAL AVERAGE

Détail des listes : 

On stocke une seule fois (on split toujours de la même façon, donc c'est commun à tous les modèles):
y_true : les vraies prédictions out-of-sample 
y_trainval_true : les vraies prédictions in-sample
dates_in : les dates pour chaque observation in-sample (utile si on veut les R² in sample)
dates_oos: dates pour chaque observations oos (utile pour récupérer les )
tickers_in : idem
tickers_oos : idem

Pour chaque modèle on stock :
- les prédictions oos : y_pred_model (ici ha)
- les prédictions in-sample : y_trainval_pred_model
- success ratio in : success_ration_in_model
- sucess ratio oos : success_ratio_oos_model
"""

#On stock : y réel et trainval réel pour calculer les r², on garde aussi les dates et tickers pour construire les portefeuilles (partie 3 results) 
y_true = []
y_trainval_true = []
dates_in = []
dates_oos = []
tickers_in = []
tickers_oos = []

y_pred_ha = []
y_trainval_pred_ha = []

r2_in_ha = []
r2_oos_ha = []

success_ratio_in_ha = []
success_ratio_oos_ha = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(preprocessed_splits, start=1):

    # In-sample
    tickers_trainval = pd.concat([x_train['Ticker'], x_val['Ticker']], ignore_index=True)
    dates_trainval = pd.concat([x_train['Date'], x_val['Date']], ignore_index=True)
    y_trainval_all = pd.concat([y_train, y_val], ignore_index=True)
    trainval = pd.DataFrame({'Ticker': tickers_trainval, 'y': y_trainval_all})

    #moyenne historique
    mean_by_ticker = trainval.groupby('Ticker')['y'].mean()
    preds_trainval = trainval['Ticker'].map(mean_by_ticker).values

    y_trainval_true.extend(trainval['y'])
    y_trainval_pred_ha.extend(preds_trainval)
    dates_in.extend(dates_trainval)
    tickers_in.extend(tickers_trainval)

    r2_in = r2(trainval['y'], preds_trainval)
    sr_in = success_ratio(trainval['y'], preds_trainval)
    r2_in_ha.append(r2_in)
    success_ratio_in_ha.append(sr_in)

    # Out-of-sample
    preds_split = [mean_by_ticker.get(tkr, np.nan) for tkr in x_test['Ticker']]
    preds_split = np.array(preds_split)
    r2_out = r2(y_test, preds_split)
    sr_out = success_ratio(y_test, preds_split)

    r2_oos_ha.append(r2_out)
    success_ratio_oos_ha.append(sr_out)

    y_pred_ha.extend(preds_split)
    y_true.extend(y_test)
    dates_oos.extend(x_test['Date'])
    tickers_oos.extend(x_test['Ticker'])

    print(f"[Split {split_idx}] R² HA IN-sample: {r2_in:.6f} | OOS: {r2_out:.6f} | SR IN: {sr_in:.3f} | SR OOS: {sr_out:.3f}")

# Conversion en array
y_trainval_true = np.array(y_trainval_true)
y_trainval_pred_ha = np.array(y_trainval_pred_ha)

#général
dates_in = np.array(dates_in)
tickers_in = np.array(tickers_in)
dates_oos = np.array(dates_oos)
tickers_oos = np.array(tickers_oos)
y_true = np.array(y_true)
y_pred_ha = np.array(y_pred_ha)

[Split 1] R² HA IN-sample: 0.022532 | OOS: -0.090013 | SR IN: 0.562 | SR OOS: 0.404
[Split 2] R² HA IN-sample: 0.014702 | OOS: 0.038366 | SR IN: 0.553 | SR OOS: 0.625
[Split 3] R² HA IN-sample: 0.017088 | OOS: 0.040437 | SR IN: 0.557 | SR OOS: 0.593
[Split 4] R² HA IN-sample: 0.018043 | OOS: -0.037014 | SR IN: 0.559 | SR OOS: 0.502
[Split 5] R² HA IN-sample: 0.016534 | OOS: 0.028351 | SR IN: 0.556 | SR OOS: 0.618
[Split 6] R² HA IN-sample: 0.016831 | OOS: 0.120348 | SR IN: 0.559 | SR OOS: 0.682
[Split 7] R² HA IN-sample: 0.018752 | OOS: 0.048575 | SR IN: 0.564 | SR OOS: 0.627
[Split 8] R² HA IN-sample: 0.019175 | OOS: -0.044497 | SR IN: 0.567 | SR OOS: 0.458
[Split 9] R² HA IN-sample: 0.018034 | OOS: 0.071978 | SR IN: 0.563 | SR OOS: 0.596
[Split 10] R² HA IN-sample: 0.019197 | OOS: 0.106359 | SR IN: 0.564 | SR OOS: 0.646
[Split 11] R² HA IN-sample: 0.020484 | OOS: -0.043278 | SR IN: 0.567 | SR OOS: 0.485
[Split 12] R² HA IN-sample: 0.018975 | OOS: 0.094262 | SR IN: 0.564 | SR OOS: 0.6

In [10]:
"""
OLS : Ordinary Least Squares : Nous détaillons ici cet algorithme, la logique étant identique pour les autres modèles.
Listes utilisées : 
- r2_in_sample_list et r2_test_list : stockent, pour chaque split, les R² in‑sample et out‑of‑sample. Elles servent à analyser
  la performance split par split et à ajuster le tuning des modèles (utile pour les modèles à hyperparamètres).
- y_true : valeurs réelles de l’equity premium sur l’ensemble.
- y_pred_ols : prédictions correspondantes du modèle OLS. 
- y_trainval : données d’entraînement (train + validation) utilisées pour l’ajustement du modèle.
- dates_ols et tickers_ols : récupérées à chaque split pour pouvoir fusionner correctement les prédictions de tous les modèles
  et s’assurer que les lignes (dates/tickers) correspondent, évitant tout mélange potentiel des prédictions.

Df et résultats en sortie : 
- df_results_ols : df contenant les prédictions du modèles ols ainsi que la date et le ticker correspondant. 
- r2_results : dictionnaire contenant le r2 ols in sample et oos
"""

#pour calculer les r² globaux 
y_trainval_ols = []

#stocke les r² par split 
r2_in_ols = []
r2_oos_ols = []
y_pred_ols = []

#sucess ratio 
success_ratio_in_ols = []
success_ratio_oos_ols = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])

    ols = LinearRegression()
    ols.fit(x_trainval, y_trainval)

    #R² in-sample
    y_trainval_pred = ols.predict(x_trainval)
    r2_in = r2(y_trainval, y_trainval_pred)
    r2_in_ols.append(r2_in)

    #R² oos
    y_test_pred = ols.predict(x_test[covariates])
    r2_out = r2(y_test, y_test_pred)
    r2_oos_ols.append(r2_out)

    #Success ratio
    sr_in = success_ratio(y_trainval, y_trainval_pred)
    sr_out = success_ratio(y_test, y_test_pred)
    success_ratio_in_ols.append(sr_in)
    success_ratio_oos_ols.append(sr_out)

    #On stocke tout dans un tableau
    y_pred_ols.append(y_test_pred)
    y_trainval_ols.append(y_trainval_pred) 

    print(f"R² in-sample : {r2_in:.6f} | R² oos : {r2_out:.6f}")
    
#On concatène les résultats de tous les splits en un df
y_pred_ols = np.concatenate(y_pred_ols)
y_trainval_ols = np.concatenate(y_trainval_ols)

 46%|████▌     | 6/13 [00:00<00:00, 51.13it/s]

R² in-sample : 0.029199 | R² oos : -0.091713
R² in-sample : 0.020787 | R² oos : 0.063573
R² in-sample : 0.024782 | R² oos : 0.048273
R² in-sample : 0.025737 | R² oos : -0.057213
R² in-sample : 0.023525 | R² oos : 0.046253
R² in-sample : 0.024027 | R² oos : 0.115323
R² in-sample : 0.025717 | R² oos : 0.006522
R² in-sample : 0.025549 | R² oos : -0.051067
R² in-sample : 0.024185 | R² oos : 0.066868
R² in-sample : 0.025134 | R² oos : 0.093977
R² in-sample : 0.026160 | R² oos : -0.041564


100%|██████████| 13/13 [00:00<00:00, 45.96it/s]

R² in-sample : 0.024563 | R² oos : 0.086490
R² in-sample : 0.026064 | R² oos : 0.009820


In [11]:
"""
PLS : Partial Least Squares
Hyperparamètres :
- k : nombre de composantes latentes, choisi pour minimiser la MSE sur la validation.
"""

dates_splits = [] 

# Pour calculer les R² globaux
y_trainval_pls = []

# Stocke les R² par split
r2_in_pls = []
r2_oos_pls = []
y_true_pls = []
y_pred_pls = []

# Success ratio
success_ratio_in_pls = []
success_ratio_oos_pls = []

# Hyperparamètres PLS
best_components_pls = []
mse_val_grids = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])

    candidate_ks = np.arange(1, 28)
    mse_val_grid = []
    best_mse = float('inf')
    best_k = None

    # Recherche du meilleur k
    for k in candidate_ks:
        pls = PLSRegression(n_components=k, scale=False)
        pls.fit(x_train[covariates], y_train)
        y_val_pred = pls.predict(x_val[covariates]).ravel()
        mse_val = mean_squared_error(y_val, y_val_pred)
        mse_val_grid.append(mse_val)

        if mse_val < best_mse:
            best_mse = mse_val
            best_k = k

    mse_val_grids.append(mse_val_grid)
    best_components_pls.append(best_k)
    
    first_date = x_test['Date'].iloc[0] #récup date split
    dates_splits.append(first_date)

    print(f"Split {split_idx} : meilleur nombre de composantes k = {best_k}")

    # Réentraîner sur train+val avec le meilleur k
    pls_final = PLSRegression(n_components=best_k, scale=False)
    pls_final.fit(x_trainval, y_trainval)

    # R² in-sample
    y_trainval_pred = pls_final.predict(x_trainval).ravel()
    r2_in = r2(y_trainval.values, y_trainval_pred)
    r2_in_pls.append(r2_in)

    # R² oos
    y_test_pred = pls_final.predict(x_test[covariates]).ravel()
    r2_out = r2(y_test, y_test_pred)
    r2_oos_pls.append(r2_out)

    # Success ratio
    sr_in = success_ratio(y_trainval.values, y_trainval_pred)
    sr_out = success_ratio(y_test.values, y_test_pred)
    success_ratio_in_pls.append(sr_in)
    success_ratio_oos_pls.append(sr_out)

    # Stockage pour global
    y_true_pls.append(y_test)
    y_pred_pls.append(y_test_pred)
    y_trainval_pls.append(y_trainval_pred)

    print(f"R² in-sample : {r2_in:.6f} | R² oos : {r2_out:.6f}")

y_true_pls = np.concatenate(y_true_pls)
y_pred_pls = np.concatenate(y_pred_pls)

y_trainval_pls = np.concatenate(y_trainval_pls)


  0%|          | 0/13 [00:00<?, ?it/s]

  8%|▊         | 1/13 [00:02<00:24,  2.05s/it]

Split 1 : meilleur nombre de composantes k = 1
R² in-sample : 0.023976 | R² oos : -0.092692


 15%|█▌        | 2/13 [00:04<00:22,  2.07s/it]

Split 2 : meilleur nombre de composantes k = 3
R² in-sample : 0.019755 | R² oos : 0.057568


...

Split 11 : meilleur nombre de composantes k = 1
R² in-sample : 0.022691 | R² oos : -0.041424


 92%|█████████▏| 12/13 [00:32<00:02,  2.98s/it]

Split 12 : meilleur nombre de composantes k = 3
R² in-sample : 0.023935 | R² oos : 0.089140


100%|██████████| 13/13 [00:35<00:00,  2.71s/it]

Split 13 : meilleur nombre de composantes k = 1
R² in-sample : 0.022834 | R² oos : 0.001464


In [12]:
from sklearn.pipeline import Pipeline

"""
PCR : Principal Component Regression
Hyperparamètre : k (nombre de composantes principales)
"""

# Pour calculer les R² globaux
y_trainval_pcr = []

# Stocke les R² par split
r2_in_pcr = []
r2_oos_pcr = []
y_true_pcr = []
y_pred_pcr = []

# Success ratio
success_ratio_in_pcr = []
success_ratio_oos_pcr = []

# Hyperparamètres spécifiques PCR
best_components_pcr = []
mse_val_grids_pcr = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])

    candidate_ks = np.arange(1, 28)
    mse_val_grid = []
    best_mse = float('inf')
    best_k = None

    # Recherche du meilleur k
    for k in candidate_ks:
        pcr_pipe = Pipeline([
            ('pca', PCA(n_components=k)),
            ('reg', LinearRegression())
        ])
        pcr_pipe.fit(x_train[covariates], y_train)
        y_val_pred = pcr_pipe.predict(x_val[covariates]).ravel()
        mse_val = mean_squared_error(y_val, y_val_pred)
        mse_val_grid.append(mse_val)

        if mse_val < best_mse:
            best_mse = mse_val
            best_k = k

    mse_val_grids_pcr.append(mse_val_grid)
    best_components_pcr.append(best_k)
    print(f"Split {split_idx} : meilleur nombre de composantes k = {best_k}")

    # Réentraîner sur train+val avec le meilleur k
    pcr_final = Pipeline([
        ('pca', PCA(n_components=best_k)),
        ('reg', LinearRegression())
    ])
    pcr_final.fit(x_trainval, y_trainval)

    # R² in-sample
    y_trainval_pred = pcr_final.predict(x_trainval).ravel()
    r2_in = r2(y_trainval.values, y_trainval_pred)
    r2_in_pcr.append(r2_in)

    # R² oos
    y_test_pred = pcr_final.predict(x_test[covariates]).ravel()
    r2_out = r2(y_test, y_test_pred)
    r2_oos_pcr.append(r2_out)

    # Success ratio
    sr_in = success_ratio(y_trainval.values, y_trainval_pred)
    sr_out = success_ratio(y_test.values, y_test_pred)
    success_ratio_in_pcr.append(sr_in)
    success_ratio_oos_pcr.append(sr_out)

    # Stockage pour global
    y_true_pcr.append(y_test)
    y_pred_pcr.append(y_test_pred)
    y_trainval_pcr.append(y_trainval_pred)

    print(f"R² in-sample : {r2_in:.6f} | R² oos : {r2_out:.6f}")

# Concaténation des résultats
y_true_pcr = np.concatenate(y_true_pcr)
y_pred_pcr = np.concatenate(y_pred_pcr)
y_trainval_pcr = np.concatenate(y_trainval_pcr)

  8%|▊         | 1/13 [00:00<00:04,  2.68it/s]

Split 1 : meilleur nombre de composantes k = 1
R² in-sample : 0.021496 | R² oos : -0.090641


 15%|█▌        | 2/13 [00:00<00:04,  2.71it/s]

Split 2 : meilleur nombre de composantes k = 6
R² in-sample : 0.016665 | R² oos : 0.052063


 23%|██▎       | 3/13 [00:01<00:03,  2.67it/s]

...

Split 11 : meilleur nombre de composantes k = 4
R² in-sample : 0.021175 | R² oos : -0.048696


 92%|█████████▏| 12/13 [00:05<00:00,  1.94it/s]

Split 12 : meilleur nombre de composantes k = 14
R² in-sample : 0.022423 | R² oos : 0.091852


100%|██████████| 13/13 [00:05<00:00,  2.24it/s]

Split 13 : meilleur nombre de composantes k = 5
R² in-sample : 0.023361 | R² oos : 0.004238


In [13]:
"""
ENet : Elastic Net

Hyperparamètres :
- lambda (alpha) : coefficient de pénalisation choisi pour minimiser la MSE
- l1_ratio fixé à 0.5
"""

# Pour calculer les R² globaux
y_trainval_en = []

# Stocke les R² par split
r2_in_en = []
r2_oos_en = []
y_pred_en = []

# Success ratio
success_ratio_in_en = []
success_ratio_oos_en = []

# Hyperparamètres spécifiques
best_lambdas = []

enet_param_grid = {
    'alpha': np.logspace(-4, 0, num=10)
}

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    best_mse = float('inf')
    best_lambda = None

    # Recherche du meilleur alpha
    for params in ParameterGrid(enet_param_grid):
        enet = ElasticNet(**params, l1_ratio=0.5, max_iter=10000)
        enet.fit(x_train[covariates], y_train)
        y_val_pred = enet.predict(x_val[covariates])
        mse = mean_squared_error(y_val, y_val_pred)
        if mse < best_mse:
            best_mse = mse
            best_lambda = params['alpha']

    best_lambdas.append(best_lambda)
    print(f"Split {split_idx} : meilleur lambda = {best_lambda}")

    # Réentraîner sur train+val
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])
    en_final = ElasticNet(alpha=best_lambda, l1_ratio=0.5, max_iter=10000)
    en_final.fit(x_trainval, y_trainval)

    # R² in-sample
    y_trainval_pred = en_final.predict(x_trainval)
    r2_in = r2(y_trainval.values, y_trainval_pred)
    r2_in_en.append(r2_in)

    # R² oos
    y_test_pred = en_final.predict(x_test[covariates])
    r2_out = r2(y_test, y_test_pred)
    r2_oos_en.append(r2_out)

    # Success ratio
    sr_in = success_ratio(y_trainval.values, y_trainval_pred)
    sr_out = success_ratio(y_test, y_test_pred)
    success_ratio_in_en.append(sr_in)
    success_ratio_oos_en.append(sr_out)

    # Stockage pour global
    y_pred_en.append(y_test_pred)
    y_trainval_en.append(y_trainval_pred)

    print(f"R² in-sample : {r2_in:.6f} | R² oos : {r2_out:.6f}")

# Concaténation des résultats
y_pred_en = np.concatenate(y_pred_en)
y_trainval_en = np.concatenate(y_trainval_en)


  8%|▊         | 1/13 [00:02<00:25,  2.10s/it]

Split 1 : meilleur lambda = 0.005994842503189409
R² in-sample : 0.019872 | R² oos : -0.081654


 15%|█▌        | 2/13 [00:08<00:50,  4.60s/it]

Split 2 : meilleur lambda = 0.016681005372000592
R² in-sample : 0.011575 | R² oos : 0.038168
Split 3 : meilleur lambda = 0.0001


 23%|██▎       | 3/13 [00:16<01:01,  6.14s/it]

...

Split 11 : meilleur lambda = 0.002154434690031882
R² in-sample : 0.022155 | R² oos : -0.035195
Split 12 : meilleur lambda = 0.000774263682681127


 92%|█████████▏| 12/13 [01:32<00:07,  7.01s/it]

R² in-sample : 0.023435 | R² oos : 0.090092
Split 13 : meilleur lambda = 0.000774263682681127


100%|██████████| 13/13 [01:37<00:00,  7.47s/it]

R² in-sample : 0.024984 | R² oos : 0.005869


In [14]:
"""
RF : Random Forest
Hyperparamètres :
- n_estimators : nombre d’arbres dans la forêt.
- max_depth : profondeur maximale de chaque arbre.
- min_samples_leaf : nombre minimal d’échantillons dans une feuille.
- max_features : nombre de variables considérées pour le split.
"""

param_grid_rf = {
    'n_estimators': [150, 200],       
    'max_depth': [3, 4, 5],
    'min_samples_leaf': [3, 5, 15],
    'max_features': ['log2', 2]
}


# Pour calculer les R² globaux
y_trainval_rf = []

# Stocke les R² par split
r2_in_rf = []
r2_oos_rf = []
y_pred_rf = []

# Success ratio
success_ratio_in_rf = []
success_ratio_oos_rf = []

# Hyperparamètres spécifiques
best_params_rf = []
mse_val_grids_rf = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    best_mse = float('inf')
    best_params = None
    mse_grid = []

    # Grid search
    for params in ParameterGrid(param_grid_rf):
        rf = RandomForestRegressor(
            **params,
            n_jobs=-1,
            random_state=0
        )
        rf.fit(x_train[covariates], y_train)
        y_val_pred = rf.predict(x_val[covariates])
        mse = mean_squared_error(y_val, y_val_pred)
        mse_grid.append((params, mse))
        if mse < best_mse:
            best_mse = mse
            best_params = params

    mse_val_grids_rf.append(mse_grid)
    best_params_rf.append(best_params)
    print(f"\nSplit {split_idx} : meilleurs params = {best_params} (MSE val = {best_mse:.6f})")

    # Train sur train+val
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])
    rf_final = RandomForestRegressor(
        **best_params,
        n_jobs=-1,
        random_state=0
    )
    rf_final.fit(x_trainval, y_trainval)

    # R² in-sample
    y_trainval_pred = rf_final.predict(x_trainval)
    r2_in = r2(y_trainval.values, y_trainval_pred)
    r2_in_rf.append(r2_in)

    # R² oos
    y_test_pred = rf_final.predict(x_test[covariates])
    r2_out = r2(y_test, y_test_pred)
    r2_oos_rf.append(r2_out)

    # Success ratio
    sr_in = success_ratio(y_trainval.values, y_trainval_pred)
    sr_out = success_ratio(y_test, y_test_pred)
    success_ratio_in_rf.append(sr_in)
    success_ratio_oos_rf.append(sr_out)

    # Stockage pour global
    y_pred_rf.append(y_test_pred)
    y_trainval_rf.append(y_trainval_pred)

    print(f"R² in-sample : {r2_in:.6f} | R² oos : {r2_out:.6f}")

# Concaténation des résultats
y_pred_rf = np.concatenate(y_pred_rf)
y_trainval_rf = np.concatenate(y_trainval_rf)


  0%|          | 0/13 [00:00<?, ?it/s]


Split 1 : meilleurs params = {'max_depth': 4, 'max_features': 'log2', 'min_samples_leaf': 15, 'n_estimators': 150} (MSE val = 0.003522)


  8%|▊         | 1/13 [00:20<04:11, 20.97s/it]

R² in-sample : 0.048442 | R² oos : -0.093527

Split 2 : meilleurs params = {'max_depth': 3, 'max_features': 2, 'min_samples_leaf': 15, 'n_estimators': 150} (MSE val = 0.015010)


 15%|█▌        | 2/13 [00:42<03:55, 21.42s/it]

...

R² in-sample : 0.046188 | R² oos : -0.032804

Split 12 : meilleurs params = {'max_depth': 5, 'max_features': 'log2', 'min_samples_leaf': 15, 'n_estimators': 200} (MSE val = 0.006090)


 92%|█████████▏| 12/13 [05:21<00:28, 28.92s/it]

R² in-sample : 0.051394 | R² oos : 0.089408

Split 13 : meilleurs params = {'max_depth': 5, 'max_features': 'log2', 'min_samples_leaf': 3, 'n_estimators': 150} (MSE val = 0.005105)


100%|██████████| 13/13 [05:51<00:00, 27.01s/it]

R² in-sample : 0.058953 | R² oos : 0.010883


In [ ]:
"""
GBRT : Gradient Boosted Regression Trees
Hyperparamètres :
- n_estimators : nombre d’arbres successifs
- learning_rate : taux d’apprentissage
- max_depth : profondeur maximale des arbres
- loss : fonction de perte
- alpha : paramètre huber
"""
param_grid_gbrt = {
    'n_estimators': [300],       
    'learning_rate': [0.01], 
    'max_depth': [2, 3],              
    'loss': ['huber'],
    'alpha': [0.9]
}


# Pour calculer les R² globaux
y_trainval_gbrt = []

# Stocke les R² par split
r2_in_gbrt = []
r2_oos_gbrt = []

y_pred_gbrt = []

# Success ratio
success_ratio_in_gbrt = []
success_ratio_oos_gbrt = []

# Hyperparamètres spécifiques
best_params_gbrt = []
mse_val_grids_gbrt = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    best_mse = float('inf')
    best_params = None
    mse_grid = []

    # Grid search
    for params in ParameterGrid(param_grid_gbrt):
        gbrt = GradientBoostingRegressor(**params, random_state=0)
        gbrt.fit(x_train[covariates], y_train)
        y_val_pred = gbrt.predict(x_val[covariates])
        mse = mean_squared_error(y_val, y_val_pred)
        mse_grid.append((params, mse))
        if mse < best_mse:
            best_mse = mse
            best_params = params

    mse_val_grids_gbrt.append(mse_grid)
    best_params_gbrt.append(best_params)
    print(f"\nSplit {split_idx} : meilleurs params = {best_params} (MSE val = {best_mse:.6f})")

    # Train sur train+val
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])
    gbrt_final = GradientBoostingRegressor(**best_params, random_state=0)
    gbrt_final.fit(x_trainval, y_trainval)

    # R² in-sample
    y_trainval_pred = gbrt_final.predict(x_trainval)
    r2_in = r2(y_trainval.values, y_trainval_pred)
    r2_in_gbrt.append(r2_in)

    # R² oos
    y_test_pred = gbrt_final.predict(x_test[covariates])
    r2_out = r2(y_test, y_test_pred)
    r2_oos_gbrt.append(r2_out)

    # Success ratio
    sr_in = success_ratio(y_trainval.values, y_trainval_pred)
    sr_out = success_ratio(y_test, y_test_pred)
    success_ratio_in_gbrt.append(sr_in)
    success_ratio_oos_gbrt.append(sr_out)

    # Stockage pour global
    y_pred_gbrt.append(y_test_pred)
    y_trainval_gbrt.append(y_trainval_pred)

    print(f"R² in-sample : {r2_in:.6f} | R² oos : {r2_out:.6f}")

# Concaténation des résultats
y_pred_gbrt = np.concatenate(y_pred_gbrt)
y_trainval_gbrt = np.concatenate(y_trainval_gbrt)


  0%|          | 0/13 [00:00<?, ?it/s]


Split 1 : meilleurs params = {'alpha': 0.9, 'learning_rate': 0.01, 'loss': 'huber', 'max_depth': 4, 'n_estimators': 300} (MSE val = 0.003479)


  8%|▊         | 1/13 [01:35<19:09, 95.82s/it]

R² in-sample : 0.091714 | R² oos : -0.077231

Split 2 : meilleurs params = {'alpha': 0.9, 'learning_rate': 0.01, 'loss': 'huber', 'max_depth': 3, 'n_estimators': 300} (MSE val = 0.014818)


 15%|█▌        | 2/13 [02:58<16:07, 87.95s/it]

...

R² in-sample : 0.030174 | R² oos : -0.025510

Split 12 : meilleurs params = {'alpha': 0.9, 'learning_rate': 0.01, 'loss': 'huber', 'max_depth': 3, 'n_estimators': 300} (MSE val = 0.006069)


 92%|█████████▏| 12/13 [25:51<02:36, 156.21s/it]

R² in-sample : 0.042909 | R² oos : 0.082681

Split 13 : meilleurs params = {'alpha': 0.9, 'learning_rate': 0.01, 'loss': 'huber', 'max_depth': 2, 'n_estimators': 300} (MSE val = 0.005140)


100%|██████████| 13/13 [28:29<00:00, 131.51s/it]

R² in-sample : 0.030416 | R² oos : 0.009648


In [ ]:
"""
XGB : XGBoost Regressor
Hyperparamètres :
- n_estimators : nombre d’arbres
- max_depth : profondeur max
- eta : learning rate
"""

param_grid_xgb = {
    'max_depth': [2, 3],
    'learning_rate': [0.01],
    'n_estimators': [200, 300, 400],
    'reg_alpha': [0, 0.1, 0.5],
    'reg_lambda': [0.5, 1, 2]
}

# Pour calculer les R² globaux
y_trainval_xgb = []
y_trainval_true = []
y_true = []
# Stocke les R² par split
r2_in_xgb = []
r2_oos_xgb = []
y_true_xgb = []
y_pred_xgb = []

# Pour les portefeuilles
dates_xgb = []
tickers_xgb = []

# Success ratio
success_ratio_in_xgb = []
success_ratio_oos_xgb = []

# Hyperparamètres spécifiques
best_params_xgb = []
mse_val_grids_xgb = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    best_mse = float('inf')
    best_params = None
    mse_grid = []

    # Grid search
    for params in ParameterGrid(param_grid_xgb):
        xgb_model = XGBRegressor(**params, random_state=0, n_jobs=-1)
        xgb_model.fit(x_train[covariates], y_train)
        y_val_pred = xgb_model.predict(x_val[covariates])
        mse = mean_squared_error(y_val, y_val_pred)
        mse_grid.append((params, mse))
        if mse < best_mse:
            best_mse = mse
            best_params = params

    mse_val_grids_xgb.append(mse_grid)
    best_params_xgb.append(best_params)
    print(f"\nSplit {split_idx} : meilleurs params = {best_params} (MSE val = {best_mse:.6f})")

    # Train sur train+val
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])
    xgb_final = XGBRegressor(**best_params, random_state=0, n_jobs=-1)
    xgb_final.fit(x_trainval, y_trainval)

    # R² in-sample
    y_trainval_pred = xgb_final.predict(x_trainval)
    
    r2_in = r2(y_trainval.values, y_trainval_pred)
    r2_in_xgb.append(r2_in)
    y_trainval_true.append(y_trainval.values)

    # R² oos
    y_test_pred = xgb_final.predict(x_test[covariates])
    r2_out = r2(y_test, y_test_pred)
    r2_oos_xgb.append(r2_out)

    # Success ratio
    sr_in = success_ratio(y_trainval.values, y_trainval_pred)
    sr_out = success_ratio(y_test, y_test_pred)
    success_ratio_in_xgb.append(sr_in)
    success_ratio_oos_xgb.append(sr_out)

    # Stockage pour global
    tickers_xgb.append(x_test["Ticker"])
    dates_xgb.append(x_test["Date"])
    y_pred_xgb.append(y_test_pred)
    y_trainval_xgb.append(y_trainval_pred)

    print(f"R² in-sample : {r2_in:.6f} | R² oos : {r2_out:.6f}")

# Concaténation des résultats
dates_xgb = np.concatenate(dates_xgb)
tickers_xgb = np.concatenate(tickers_xgb)
y_pred_xgb = np.concatenate(y_pred_xgb)
y_trainval_xgb = np.concatenate(y_trainval_xgb)
y_trainval_true = np.concatenate(y_trainval_true)
y_true = np.concatenate(y_true)
df_results_xgb = pd.DataFrame({
    "Date": dates_xgb,
    "Ticker": tickers_xgb,
    "y_pred_xgb": y_pred_xgb,
})

r2_in_xgb = r2(y_trainval_true, y_trainval_xgb)
r2_oos_xgb = r2(y_true, y_pred_xgb)

print("XGB   : In-sample =", r2_in_xgb,  "| OOS =", r2_oos_xgb)


  0%|          | 0/13 [00:00<?, ?it/s]


Split 1 : meilleurs params = {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 200, 'reg_alpha': 0.5, 'reg_lambda': 2} (MSE val = 0.003570)


  8%|▊         | 1/13 [00:42<08:28, 42.36s/it]

R² in-sample : 0.052651 | R² oos : -0.110629

Split 2 : meilleurs params = {'learning_rate': 0.01, 'max_depth': 2, 'n_estimators': 200, 'reg_alpha': 0.5, 'reg_lambda': 2} (MSE val = 0.015174)


 15%|█▌        | 2/13 [01:26<07:54, 43.16s/it]

...

 85%|████████▍ | 11/13 [09:50<01:46, 53.06s/it]

R² in-sample : 0.040839 | R² oos : -0.028819

Split 12 : meilleurs params = {'learning_rate': 0.01, 'max_depth': 2, 'n_estimators': 400, 'reg_alpha': 0, 'reg_lambda': 0.5} (MSE val = 0.006086)


 92%|█████████▏| 12/13 [10:50<00:55, 55.27s/it]

R² in-sample : 0.034731 | R² oos : 0.089543

Split 13 : meilleurs params = {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 400, 'reg_alpha': 0, 'reg_lambda': 0.5} (MSE val = 0.005087)


100%|██████████| 13/13 [11:51<00:00, 54.73s/it]

II. METRIQUES 

In [ ]:
#R2 

# OLS
r2_in_ols = r2(y_trainval_true, y_trainval_ols)
r2_oos_ols = r2(y_true, y_pred_ols)

r2_in_pls = r2(y_trainval_true, y_trainval_pls)
r2_oos_pls = r2(y_true, y_pred_pls)

r2_in_pcr = r2(y_trainval_true, y_trainval_pcr)
r2_oos_pcr = r2(y_true, y_pred_pcr)

r2_in_en = r2(y_trainval_true, y_trainval_en)
r2_oos_en = r2(y_true, y_pred_en)

r2_in_rf = r2(y_trainval_true, y_trainval_rf)
r2_oos_rf = r2(y_true, y_pred_rf)

r2_in_gbrt = r2(y_trainval_true, y_travail_gbrt)
r2_oos_gbrt = r2(y_true, y_pred_gbrt)

r2_in_xgb = r2(y_trainval_true, y_trainval_xgb)
r2_oos_xgb = r2(y_true, y_pred_xgb)

#XGB 

# Affichage
print("OLS   : In-sample =", r2_in_ols,  "| OOS =", r2_oos_ols)
print("PLS   : In-sample =", r2_in_pls,  "| OOS =", r2_oos_pls)
print("PCR   : In-sample =", r2_in_pcr,  "| OOS =", r2_oos_pcr)
print("Enet  : In-sample =", r2_in_en,   "| OOS =", r2_oos_en)
print("Rf  : In-sample =", r2_in_rf,   "| OOS =", r2_oos_rf)
print("GBRT  : In-sample =", r2_in_gbrt,   "| OOS =", r2_oos_gbrt)
print("XGB : In-sample =", r2_in_xgb,   "| OOS =", r2_oos_xgb)")

ValueError: operands could not be broadcast together with shapes (9290,) (3580,) 

NameError: name 'y_trainval_true' is not defined

In [ ]:
# GBRT
r2_in_gbrt = r2(y_trainval_true, y_trainval_gbrt)
r2_oos_gbrt = r2(y_true, y_pred_gbrt)

# XGB
r2_in_xgb = r2(y_trainval_true, y_trainval_xgb)
r2_oos_xgb = r2(y_true, y_pred_xgb)

print("GBRT  : In-sample =", r2_in_gbrt, "| OOS =", r2_oos_gbrt)
print("XGB   : In-sample =", r2_in_xgb,  "| OOS =", r2_oos_xgb)